In [27]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import librosa as lb
import librosa.display
import matplotlib.pyplot as plt
import soundfile as sf
from pathlib import Path
from tqdm import tqdm

In [28]:
clean_testset_chunks = np.load("data/processed/clean_testset_chunks.npy")
clean_trainset_chunks = np.load("data/processed/clean_trainset_chunks.npy")
noisy_testset_chunks = np.load("data/processed/noisy_testset_chunks.npy")
noisy_trainset_chunks = np.load("data/processed/noisy_trainset_chunks.npy")
print(clean_testset_chunks.shape)
print(clean_trainset_chunks.shape)
print(noisy_testset_chunks.shape)
print(noisy_trainset_chunks.shape)


(2470, 16000)
(39630, 16000)
(2470, 16000)
(39630, 16000)


In [29]:
clean_testset_amp = np.load("data/processed/clean_testset_amp.npy")
clean_trainset_amp = np.load("data/processed/clean_trainset_amp.npy")
noisy_testset_amp = np.load("data/processed/noisy_testset_amp.npy")
noisy_trainset_amp = np.load("data/processed/noisy_trainset_amp.npy")

print(clean_testset_amp.shape,
clean_trainset_amp.shape,
noisy_testset_amp.shape,
noisy_trainset_amp.shape)

(2470, 1025, 32) (39630, 1025, 32) (2470, 1025, 32) (39630, 1025, 32)


In [30]:
torch_clean_testset_amp = (torch.from_numpy(clean_testset_amp)).float()
torch_clean_trainset_amp = (torch.from_numpy(clean_trainset_amp)).float()
torch_noisy_testset_amp = (torch.from_numpy(noisy_testset_amp)).float()
torch_noisy_trainset_amp = (torch.from_numpy(noisy_trainset_amp)).float()

print(torch_clean_testset_amp.shape, torch_clean_trainset_amp.shape, torch_noisy_testset_amp.shape, torch_noisy_trainset_amp.shape )
print(torch_clean_testset_amp.dtype, torch_clean_trainset_amp.dtype, torch_noisy_testset_amp.dtype, torch_noisy_trainset_amp.dtype )

torch.Size([2470, 1025, 32]) torch.Size([39630, 1025, 32]) torch.Size([2470, 1025, 32]) torch.Size([39630, 1025, 32])
torch.float32 torch.float32 torch.float32 torch.float32


In [31]:
train_dataset = TensorDataset(torch_noisy_trainset_amp,torch_clean_trainset_amp )
test_dataset = TensorDataset(torch_noisy_testset_amp ,torch_clean_testset_amp )
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [32]:

class ConvBlock(nn.Module):
  def __init__(self,in_channels,out_channels):
    super().__init__()
    self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu1 = nn.ReLU()
    self.conv2 = nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.relu2 = nn.ReLU()
  def forward(self, x):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu1(x)
    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu2(x)
    return x

In [33]:
block1 = ConvBlock(1, 32)
sample = torch_noisy_trainset_amp[10]
sample = sample.unsqueeze(0).unsqueeze(0)
output1 = block1(sample)
output1.shape

torch.Size([1, 32, 1025, 32])

In [34]:
pool = nn.MaxPool2d(kernel_size=2)
result = pool(output1)
result.shape




torch.Size([1, 32, 512, 16])

In [35]:
block2 = ConvBlock(32, 64)
output2 = block2(result)
output2.shape

torch.Size([1, 64, 512, 16])

In [36]:
pool2 = nn.MaxPool2d(kernel_size=2)
result2 = pool2(output2)
result2.shape

torch.Size([1, 64, 256, 8])

In [37]:
bottleneck = ConvBlock(64,128)
output3 = bottleneck(result2)
output3.shape



torch.Size([1, 128, 256, 8])

In [38]:
upsample = nn.Upsample(scale_factor=2)
output4 = upsample(output3)
output4.shape

torch.Size([1, 128, 512, 16])

In [39]:
combined = torch.cat([output4, output2], dim=1)
combined.shape

torch.Size([1, 192, 512, 16])

In [40]:
decoder_block1 = ConvBlock(192, 64)
output5 = decoder_block1(combined)
output5.shape

torch.Size([1, 64, 512, 16])

In [41]:
upsample2 = nn.Upsample(size=(1025, 32))
output6 = upsample2(output5)
output6.shape

torch.Size([1, 64, 1025, 32])

In [42]:
combined2 = torch.cat([output6, output1], dim=1)
combined2.shape


torch.Size([1, 96, 1025, 32])

In [43]:
decoder_block2 = ConvBlock(96, 32)
output7 = decoder_block2(combined2)
output7.shape


torch.Size([1, 32, 1025, 32])

In [45]:
final_conv = nn.Conv2d(in_channels=32, out_channels=1, kernel_size=1)
prediction = final_conv(output7)
prediction.shape


torch.Size([1, 1, 1025, 32])